In [ ]:
import pandas as pd
import numpy as np
import pickle
import re
import os
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import FeatureUnion

print("1. A carregar, limpar e balancear os dados...")
#Load dataset
df = pd.read_csv('dataset_final.csv', sep=';')

#Corrige linhas de cabeçalho
df = df[df['Label'] != 'Label']
df['Text'] = df['Text'].str.replace(r'<ref.*?>.*?</ref>|<ref.*?>', '', regex=True)

#Balance dataset
min_count = df['Label'].value_counts().min()
df_balanced = df.groupby('Label').sample(n=min_count, random_state=42)
df_balanced = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)

textos = df_balanced['Text'].values
labels = df_balanced['Label'].values

print(f"Dataset pronto! Total de linhas perfeitamente balanceadas: {len(df_balanced)}")

print("\n2. A executar TF-IDF (FeatureUnion: Caracteres + Palavras)...")

# 2A. O Vetorizador de Caracteres (Captura estrutura, pontuação, sufixos)
char_vect = TfidfVectorizer(
    analyzer='char', 
    ngram_range=(2, 5), 
    max_features=8000,   
    sublinear_tf=True
)

# 2B. O Vetorizador de Palavras (Captura vocabulário e termos específicos de IA)
word_vect = TfidfVectorizer(
    analyzer='word', 
    ngram_range=(1, 3), 
    max_features=4000,   
    lowercase=True,
    sublinear_tf=True
)

vectorizer = FeatureUnion([
    ("char", char_vect),
    ("word", word_vect)
])

X = vectorizer.fit_transform(textos).toarray()

os.makedirs('Subm2', exist_ok=True)
with open('Subm2/tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(vectorizer, f)

PROFESSOR_MAP = {
    'Human': 0,
    'OpenAI': 1,
    'Google': 2,
    'Meta': 3,
    'Anthropic': 4
}
y = np.array([PROFESSOR_MAP[label] for label in labels])

#Separar 10% do Total para o Conjunto de Teste Final (Teste = 10% do Total)
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.10, random_state=42, stratify=y
)

#Separar 11% do Conjunto de Treino Total para o Conjunto de Validação (Validação = 11% do Treino Total, Teste = 10% do Total)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.11, random_state=42, stratify=y_train_full
)

#Guardar os Arrays de Ajuste (Treino & Validação)
np.save('X_train.npy', X_train)
np.save('y_train.npy', y_train)
np.save('X_val.npy', X_val)
np.save('y_val.npy', y_val)

#Guardar os Arrays de Teste Finais (Teste)
np.save('X_test.npy', X_test)
np.save('y_test.npy', y_test)

#Guardar os arrays "Full Train"
np.save('X_train_full.npy', X_train_full)
np.save('y_train_full.npy', y_train_full)

print(f"\nSucesso! Dados particionados corretamente:")
print(f"  -> X_train (para ajuste):      {X_train.shape}")
print(f"  -> X_val (para validação):    {X_val.shape}")
print(f"  -> X_train_full (para final):  {X_train_full.shape}")
print(f"  -> X_test (avaliação final): {X_test.shape}")

1. Loading, Cleaning, and Balancing data...
Dataset ready! Total perfectly balanced rows: 4355

2. Running TF-IDF (FeatureUnion: Char + Word)...

3. Encoding Labels (Using Professor's Custom Map)...
Mapped successfully using custom integer dictionary!

4. Splitting Data According to Professor's Rules...

Success! Data successfully partitioned:
  -> X_train (for tuning):      (3527, 12000)
  -> X_val (for validation):    (392, 12000)
  -> X_train_full (for final):  (3919, 12000)
  -> X_test (final evaluation): (436, 12000)
